# 01 — From token IDs to position-aware states

A token ID is an address, not a numerical measurement of meaning. The same embedding row can be used at several positions; a position embedding gives those occurrences different starting states.

We will inspect $X_{b,t}=E[\mathrm{id}_{b,t}]+P[t]$, where IDs have shape $[B,T]$ and states $[B,T,D]$. No contextual mixing has happened yet.

Prerequisites: Chapters 2–4. CPU only; no tokenizer or model download. This is the first of eleven Chapter 5 sessions.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [1]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


CPU reference environment: 2.13.0+cu130


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from dongxi_llms import decoder_visuals as viz
def show_visual(figure):
    display(figure)
    plt.close(figure)


## Architecture map — your location in the model

The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The baseline route adds learned position embeddings before the blocks.

![Architecture map — your location in the model. The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The baseline route adds learned position embeddings before the blocks.](../figures/chapter-05/day-05-01_embeddings_and_positions-architecture-map.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
from dongxi_llms import decoder_architecture as architecture
show_visual(architecture.model_map(focus='embeddings', modern=False))

## Two lookup branches meet at addition

Token IDs select E rows; position indices select P rows. Position vectors broadcast over the batch, and addition preserves model width D.

![Two lookup branches meet at addition. Token IDs select E rows; position indices select P rows. Position vectors broadcast over the batch, and addition preserves model width D.](../figures/chapter-05/day-05-01_embeddings_and_positions-architecture-detail.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
show_visual(architecture.embedding_detail())

## 1. Lookup without hiding the operation

If the same token occurs twice, must its lookup vectors match? Implement lookup by indexing an embedding table and compare it to nn.Embedding.

**Your prediction:** _Write it here before running the reference._

In [3]:
ids = torch.tensor([[2, 5, 2, 7], [5, 2, 7, 2]])
embedding = nn.Embedding(16, 8).double()
# Your implementation: indexed = ...


### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [4]:
indexed = embedding.weight[ids]
reference = embedding(ids)
close(indexed, reference)
close(indexed[0, 0], indexed[0, 2])
print("IDs:", ids.shape, "states:", indexed.shape)
print("Repeated token vectors:", indexed[0, [0, 2]])

IDs: torch.Size([2, 4]) states: torch.Size([2, 4, 8])
Repeated token vectors: tensor([[-0.2502, -0.2149, -0.8013,  0.2977, -0.7333,  0.6690,  0.5400,  0.9359],
        [-0.2502, -0.2149, -0.8013,  0.2977, -0.7333,  0.6690,  0.5400,  0.9359]],
       dtype=torch.float64, grad_fn=<IndexBackward0>)


### Why this works

The repeated ID selects the same row. IDs 2 and 5 are categorical addresses; their numerical distance is not a semantic distance.

### Visual explanation — Follow one ID into the table

The green outlines identify repeated selections of the same stored row. Colors encode coordinate values, not semantic similarity.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![Follow one ID into the table. The green outlines identify repeated selections of the same stored row. Colors encode coordinate values, not semantic similarity.](../figures/chapter-05/day-05-01_embeddings_and_positions-visual-lookup.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.embedding_lookup(embedding.weight, ids[0]))

## 2. Give occurrences different positions

Add learned absolute positions. Predict what changes if we move the position indices by one while keeping token IDs fixed.

**Your prediction:** _Write it here before running the reference._

In [ ]:
position = nn.Embedding(12, 8).double()
# Your implementation: states = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
positions = torch.arange(ids.shape[1])
states = indexed + position(positions)[None]
shifted = indexed + position(positions + 1)[None]
close(states - indexed, position(positions)[None].expand_as(states))
assert not torch.allclose(states[0, 0], states[0, 2])
print("Same token, different positions:", states[0, [0, 2]])
print("Position-offset change norm:", float((shifted-states).norm().detach()))

### Why this works

Position vectors broadcast across the batch, not across the feature dimension. Addition keeps width D; it does not concatenate two vectors. Learned positions are parameters, and this offset intervention changes the model input.

### Visual explanation — See token + position = starting state

Read left to right on one shared color scale. The token rows can repeat while the position vectors differ; addition changes the initial states.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![See token + position = starting state. Read left to right on one shared color scale. The token rows can repeat while the position vectors differ; addition changes the initial states.](../figures/chapter-05/day-05-01_embeddings_and_positions-visual-positions.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.matrices([indexed[0], position(positions), states[0]], ['Token lookup', 'Position lookup', 'Their sum'], 'Same token identity, different position-aware states'))

## 3. Watch shared embedding rows accumulate gradients

Backpropagate a sum of selected features. Which rows receive lookup-path gradients? What happens when a token repeats?

**Your prediction:** _Write it here before running the reference._

In [7]:
embedding.zero_grad(set_to_none=True)
# Your implementation: loss = embedding(ids).sum(); ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [8]:
embedding(ids).sum().backward()
counts = torch.bincount(ids.flatten(), minlength=16).to(DTYPE)
close(embedding.weight.grad, counts[:, None].expand(16, 8))
print("Token occurrence counts:", counts.tolist())
print("Gradient of row 2:", embedding.weight.grad[2])
print("Unused-row gradient:", embedding.weight.grad[0])

Token occurrence counts: [0.0, 0.0, 4.0, 0.0, 0.0, 2.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Gradient of row 2: tensor([4., 4., 4., 4., 4., 4., 4., 4.], dtype=torch.float64)
Unused-row gradient: tensor([0., 0., 0., 0., 0., 0., 0., 0.], dtype=torch.float64)


### Why this works

Each occurrence contributes to the same stored row. Here the artificial sum objective makes each contribution one. In a tied language model the output-head path can also update rows absent from the input, so this result applies specifically to lookup.

### Visual explanation — See contributions accumulate

Bars show the measured gradient of coordinate 0 in each embedding row; crosses show occurrence counts. They coincide for this notebook’s sum objective, not for arbitrary prediction losses.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![See contributions accumulate. Bars show the measured gradient of coordinate 0 in each embedding row; crosses show occurrence counts. They coincide for this notebook’s sum objective, not for arbitrary prediction losses.](../figures/chapter-05/day-05-01_embeddings_and_positions-visual-lookup-gradients.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.lookup_gradients(ids, embedding.weight.grad))

## 4. Separate position signals from causal structure

Disable absolute positions in a tiny model without changing its token embeddings. Does that remove the causal mask too?

**Your prediction:** _Write it here before running the reference._

In [9]:
model = TinyDecoder().double().eval()
batch, _ = teaching_batch()
# Predict before inspecting the outputs.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [10]:
without_positions = copy.deepcopy(model)
with torch.no_grad():
    without_positions.position.weight.zero_()
    original = model(batch)
    ablated = without_positions(batch)
    changed = batch.clone(); changed[:, 4:] = 0
    close(without_positions(changed)[:, :4], ablated[:, :4])
print("Position-ablation output change:", float((original-ablated).norm()))

Position-ablation output change: 0.8170118536529617


### Why this works

Zeroing P removes explicit learned positions but does not remove the causal graph. A causal model without position embeddings is not generally order-blind. This untrained intervention measures a numerical change, not a quality improvement.

## Takeaway and evidence boundary

You can now explain the input to the first block. Next: split that state into multiple learned retrieval views. Evidence: lookup identities and gradients, not trained semantic structure.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.